# WorldSim Model Comparison — Base vs Fine-tuned × 0.8B vs 2B
Visual side-by-side comparison across 20 task types


## 1. Setup & Model Inventory


In [ ]:

from pathlib import Path
import json
import os
import re
import subprocess
import sys
import time

cwd = Path.cwd()
repo_marker = cwd / 'training/run_qlora_train.py'
notebook_marker = cwd / 'notebooks/dgx_spark_model_comparison.ipynb'
assert repo_marker.exists() and notebook_marker.exists(), (
    'Run this notebook from the repository root: worldsim-training'
)
REPO_ROOT = cwd
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

GGUF_DIR = REPO_ROOT / 'artifacts' / 'gguf'
GGUF_DIR.mkdir(parents=True, exist_ok=True)
LLAMA_SERVER = REPO_ROOT / 'tools' / 'llama.cpp' / 'build' / 'bin' / 'llama-server'
LLAMA_QUANTIZE = REPO_ROOT / 'tools' / 'llama.cpp' / 'build' / 'bin' / 'llama-quantize'
CONVERT_SCRIPT = REPO_ROOT / 'tools' / 'llama.cpp' / 'convert_hf_to_gguf.py'

FT_08B = GGUF_DIR / 'worldsim-v3-qwen3.5-0.8b-q4_k_m.gguf'
FT_2B = GGUF_DIR / 'worldsim-v3-qwen3.5-2b-q4_k_m.gguf'
BASE_08B = GGUF_DIR / 'qwen3.5-0.8b-base-q4_k_m.gguf'
BASE_2B = GGUF_DIR / 'qwen3.5-2b-base-q4_k_m.gguf'

MODELS = {}
if FT_08B.exists():
    MODELS['ft_08b'] = {'path': FT_08B, 'label': '0.8B Fine-tuned v3', 'size_mb': FT_08B.stat().st_size / 1e6}
if FT_2B.exists():
    MODELS['ft_2b'] = {'path': FT_2B, 'label': '2B Fine-tuned v3', 'size_mb': FT_2B.stat().st_size / 1e6}
if BASE_08B.exists():
    MODELS['base_08b'] = {'path': BASE_08B, 'label': '0.8B Base (no FT)', 'size_mb': BASE_08B.stat().st_size / 1e6}
if BASE_2B.exists():
    MODELS['base_2b'] = {'path': BASE_2B, 'label': '2B Base (no FT)', 'size_mb': BASE_2B.stat().st_size / 1e6}

print('=== Model Inventory ===')
for key, info in MODELS.items():
    print(f"  {info['label']:.<30} {info['size_mb']:.0f} MB ✅")

missing_base = []
if not BASE_08B.exists():
    missing_base.append(('Qwen/Qwen3.5-0.8B-Base', BASE_08B, 'qwen3.5-0.8b-base'))
if not BASE_2B.exists():
    missing_base.append(('Qwen/Qwen3.5-2B-Base', BASE_2B, 'qwen3.5-2b-base'))

if missing_base:
    print()
    print(f"⚠️ {len(missing_base)} base model GGUF(s) missing — will convert in next cell")
else:
    print()
    print('✅ All 4 models ready')


## 2. Convert Base Models to GGUF (if needed)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def convert_base_to_gguf(model_name: str, output_gguf: Path, short_name: str) -> None:
    if output_gguf.exists():
        print(f"  {output_gguf.name} already exists ✅")
        return

    work_dir = REPO_ROOT / 'artifacts' / 'base_models' / short_name
    work_dir.mkdir(parents=True, exist_ok=True)
    fp16_gguf = work_dir / f'{short_name}-f16.gguf'
    hf_dir = work_dir / 'hf'

    if not (hf_dir / 'config.json').exists():
        print(f"  Downloading {model_name}...", flush=True)
        model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16, device_map='auto')
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        hf_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(str(hf_dir))
        tokenizer.save_pretrained(str(hf_dir))
        del model
        torch.cuda.empty_cache()
        print('  Downloaded ✅', flush=True)
    else:
        print('  HF model cached ✅', flush=True)

    if not fp16_gguf.exists():
        print('  Converting to GGUF FP16...', flush=True)
        result = subprocess.run(
            [sys.executable, str(CONVERT_SCRIPT), str(hf_dir), '--outfile', str(fp16_gguf), '--outtype', 'f16'],
            capture_output=True,
            text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(f"GGUF conversion failed: {result.stderr[-500:]}")
        print(f"  FP16: {fp16_gguf.stat().st_size / 1e6:.0f} MB ✅", flush=True)

    print('  Quantizing to Q4_K_M...', flush=True)
    result = subprocess.run(
        [str(LLAMA_QUANTIZE), str(fp16_gguf), str(output_gguf), 'Q4_K_M'],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"Quantization failed: {result.stderr[-500:]}")
    print(f"  Q4_K_M: {output_gguf.stat().st_size / 1e6:.0f} MB ✅", flush=True)


for model_name, output_path, short_name in missing_base:
    print()
    print(f'=== Converting {model_name} ===')
    convert_base_to_gguf(model_name, output_path, short_name)

for key, path, label in [
    ('base_08b', BASE_08B, '0.8B Base (no FT)'),
    ('base_2b', BASE_2B, '2B Base (no FT)'),
]:
    if path.exists() and key not in MODELS:
        MODELS[key] = {'path': path, 'label': label, 'size_mb': path.stat().st_size / 1e6}

print()
print(f'✅ {len(MODELS)} models ready')
for key, info in sorted(MODELS.items()):
    print(f"  {info['label']:.<30} {info['size_mb']:.0f} MB")


## 3. Server Helper & Test Prompts


In [ ]:

import json
import urllib.request
from collections import defaultdict

SERVER_PORT = 8090
SERVER_URL = f'http://127.0.0.1:{SERVER_PORT}'


def start_server(model_path: Path, ctx_size: int = 2048, n_gpu_layers: int = 99):
    args = [
        str(LLAMA_SERVER), '-m', str(model_path),
        '--host', '127.0.0.1', '--port', str(SERVER_PORT),
        '-c', str(ctx_size), '-np', '1', '-ngl', str(n_gpu_layers),
        '-fa', 'on', '--jinja', '--no-webui',
        '--chat-template-kwargs', '{"enable_thinking": false}',
        '-ctk', 'q8_0', '-ctv', 'q8_0',
    ]
    proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for attempt in range(120):
        time.sleep(0.5)
        try:
            resp = urllib.request.urlopen(f'{SERVER_URL}/health', timeout=2)
            data = json.loads(resp.read())
            if data.get('status') == 'ok':
                return proc
        except Exception:
            pass
        if proc.poll() is not None:
            stderr = proc.stderr.read().decode(errors='ignore')[-300:]
            raise RuntimeError(f'Server died: {stderr}')
    proc.kill()
    raise RuntimeError('Server startup timeout')


def stop_server(proc) -> None:
    if proc and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except Exception:
            proc.kill()
            proc.wait()


def strip_think(text: str) -> str:
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()


def query_model(messages, response_format=None, max_tokens: int = 256, temperature: float = 0):
    payload = {
        'messages': messages,
        'max_tokens': max_tokens,
        'temperature': temperature,
        'stream': False,
    }
    if response_format:
        payload['response_format'] = response_format

    req = urllib.request.Request(
        f'{SERVER_URL}/v1/chat/completions',
        data=json.dumps(payload).encode(),
        headers={'Content-Type': 'application/json'},
    )
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            data = json.loads(resp.read())
        content = data['choices'][0]['message']['content']
        return strip_think(content)
    except Exception as exc:
        return f'ERROR: {exc}'


SYS_L3 = 'You are WorldSim logic assistant. Output JSON only. Follow [TEMP], [STRESS], [WORLD] context. Respect enum values and numeric ranges exactly.'
SYS_L4 = '너는 WorldSim 서사 도우미다. [TEMP], [STRESS], [WORLD]와 지정된 register를 반영해 JSON으로만 답하라.'
SYS_L5 = '너는 WorldSim 신탁 해석 도우미다. JSON으로만 답하라.'

EMOTIONS = ['joy', 'sadness', 'fear', 'anger', 'trust', 'disgust', 'surprise', 'anticipation']
SPEAKER_ROLES = ['elder', 'hunter', 'shaman', 'warrior', 'healer', 'gatherer', 'craftsman', 'chief', 'scout', 'observer']

TASK_E_SCHEMA = {
    'type': 'json_schema',
    'json_schema': {
        'name': 'task_e',
        'strict': True,
        'schema': {
            'type': 'object',
            'additionalProperties': False,
            'required': ['action_id', 'confidence', 'hint', 'personality_reasoning', 'temperament_factor'],
            'properties': {
                'action_id': {'type': 'integer', 'enum': [0, 1, 2, 3, 4]},
                'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
                'hint': {'type': 'string'},
                'personality_reasoning': {'type': 'string', 'enum': ['novelty_seeking', 'harm_avoidance', 'reward_dependence', 'persistence']},
                'temperament_factor': {'type': 'string'},
            },
        },
    },
}
TASK_B_SCHEMA = {
    'type': 'json_schema',
    'json_schema': {
        'name': 'task_b',
        'strict': True,
        'schema': {
            'type': 'object',
            'additionalProperties': False,
            'required': ['text_ko', 'text_en', 'register', 'emotion_expressed', 'intensity', 'mimetics', 'temperament_influence'],
            'properties': {
                'text_ko': {'type': 'string'},
                'text_en': {'type': 'string'},
                'register': {'type': 'string', 'enum': ['haera']},
                'emotion_expressed': {'type': 'string', 'enum': EMOTIONS},
                'intensity': {'type': 'number'},
                'mimetics': {'type': 'array', 'items': {'type': 'string'}},
                'temperament_influence': {'type': 'string'},
            },
        },
    },
}
TASK_C_SCHEMA = {
    'type': 'json_schema',
    'json_schema': {
        'name': 'task_c',
        'strict': True,
        'schema': {
            'type': 'object',
            'additionalProperties': False,
            'required': ['speech_ko', 'speech_en', 'register', 'emotion_expressed', 'speaker_role', 'temperament_tone'],
            'properties': {
                'speech_ko': {'type': 'string'},
                'speech_en': {'type': 'string'},
                'register': {'type': 'string', 'enum': ['haera']},
                'emotion_expressed': {'type': 'string', 'enum': EMOTIONS},
                'speaker_role': {'type': 'string', 'enum': SPEAKER_ROLES},
                'temperament_tone': {'type': 'string'},
            },
        },
    },
}
TASK_F_SCHEMA = {
    'type': 'json_schema',
    'json_schema': {
        'name': 'task_f',
        'strict': True,
        'schema': {
            'type': 'object',
            'additionalProperties': False,
            'required': ['emotion', 'intensity', 'cause', 'previous_emotion', 'transition_type', 'temperament_amplifier'],
            'properties': {
                'emotion': {'type': 'string', 'enum': EMOTIONS},
                'intensity': {'type': 'number', 'minimum': 0, 'maximum': 1},
                'cause': {'type': 'string'},
                'previous_emotion': {'type': 'string', 'enum': EMOTIONS},
                'transition_type': {'type': 'string', 'enum': ['gradual', 'sudden', 'sustained']},
                'temperament_amplifier': {'type': 'string'},
            },
        },
    },
}
TASK_G_SCHEMA = {
    'type': 'json_schema',
    'json_schema': {
        'name': 'task_g',
        'strict': True,
        'schema': {
            'type': 'object',
            'additionalProperties': False,
            'required': ['interpretation_ko', 'interpretation_en', 'action_tendency', 'confidence', 'register', 'misinterpretation_type', 'temperament_bias'],
            'properties': {
                'interpretation_ko': {'type': 'string'},
                'interpretation_en': {'type': 'string'},
                'action_tendency': {'type': 'string', 'enum': ['mobilize', 'defend', 'wait', 'retreat', 'celebrate', 'mourn']},
                'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
                'register': {'type': 'string', 'enum': ['haera', 'hao', 'hae']},
                'misinterpretation_type': {'type': 'string', 'enum': ['overconfident_literal', 'cautious_reversal', 'optimistic_expansion', 'passive_deferral', 'symbolic_abstraction']},
                'temperament_bias': {'type': 'string'},
            },
        },
    },
}
TASK_M_SCHEMA = {
    'type': 'json_schema',
    'json_schema': {
        'name': 'task_m',
        'strict': True,
        'schema': {
            'type': 'object',
            'additionalProperties': False,
            'required': ['decision_id', 'confidence', 'dissent_risk', 'reasoning', 'resource_commitment', 'timeline'],
            'properties': {
                'decision_id': {'type': 'integer', 'enum': [0, 1, 2, 3, 4]},
                'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
                'dissent_risk': {'type': 'number', 'minimum': 0, 'maximum': 1},
                'reasoning': {'type': 'string'},
                'resource_commitment': {'type': 'string', 'enum': ['food', 'tools', 'labor', 'weapons', 'none']},
                'timeline': {'type': 'string', 'enum': ['immediate', 'delayed', 'conditional']},
            },
        },
    },
}
TASK_O_SCHEMA = {
    'type': 'json_schema',
    'json_schema': {
        'name': 'task_o',
        'strict': True,
        'schema': {
            'type': 'object',
            'additionalProperties': False,
            'required': ['public_claim', 'private_truth', 'deception_style', 'lie_degree', 'detection_risk', 'confidence'],
            'properties': {
                'public_claim': {'type': 'string'},
                'private_truth': {'type': 'string'},
                'deception_style': {'type': 'string', 'enum': ['evasion', 'half_truth', 'outright_lie', 'exaggeration', 'omission']},
                'lie_degree': {'type': 'number', 'minimum': 0, 'maximum': 1},
                'detection_risk': {'type': 'number', 'minimum': 0, 'maximum': 1},
                'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
            },
        },
    },
}
TASK_P_SCHEMA = {
    'type': 'json_schema',
    'json_schema': {
        'name': 'task_p',
        'strict': True,
        'schema': {
            'type': 'object',
            'additionalProperties': False,
            'required': ['retold_version', 'distortion_type', 'added_detail', 'dropped_detail', 'emotional_charge'],
            'properties': {
                'retold_version': {'type': 'string'},
                'distortion_type': {'type': 'string', 'enum': ['exaggeration', 'minimization', 'malicious_twist', 'emotional_coloring', 'detail_invention', 'source_confusion', 'faithful']},
                'added_detail': {'type': 'string'},
                'dropped_detail': {'type': 'string'},
                'emotional_charge': {'type': 'number', 'minimum': -1, 'maximum': 1},
            },
        },
    },
}
TASK_Q_SCHEMA = {
    'type': 'json_schema',
    'json_schema': {
        'name': 'task_q',
        'strict': True,
        'schema': {
            'type': 'object',
            'additionalProperties': False,
            'required': ['trauma_response', 'behavioral_change', 'trigger_situation', 'intensity', 'duration', 'coping_mechanism'],
            'properties': {
                'trauma_response': {'type': 'string', 'enum': ['avoidance', 'overprotection', 'aggression', 'withdrawal', 'hypervigilance', 'ritual_coping', 'resilience']},
                'behavioral_change': {'type': 'string'},
                'trigger_situation': {'type': 'string'},
                'intensity': {'type': 'number', 'minimum': 0, 'maximum': 1},
                'duration': {'type': 'string', 'enum': ['short_term', 'long_term', 'permanent']},
                'coping_mechanism': {'type': 'string'},
            },
        },
    },
}
TASK_R_SCHEMA = {
    'type': 'json_schema',
    'json_schema': {
        'name': 'task_r',
        'strict': True,
        'schema': {
            'type': 'object',
            'additionalProperties': False,
            'required': ['action', 'counter_give', 'counter_want', 'reasoning', 'emotional_state', 'walk_away_threshold'],
            'properties': {
                'action': {'type': 'string', 'enum': ['accept', 'counter_offer', 'reject', 'walk_away', 'stall', 'bluff']},
                'counter_give': {'type': 'string'},
                'counter_want': {'type': 'string'},
                'reasoning': {'type': 'string'},
                'emotional_state': {'type': 'string', 'enum': EMOTIONS},
                'walk_away_threshold': {'type': 'number', 'minimum': 0, 'maximum': 1},
            },
        },
    },
}
TASK_T_SCHEMA = {
    'type': 'json_schema',
    'json_schema': {
        'name': 'task_t',
        'strict': True,
        'schema': {
            'type': 'object',
            'additionalProperties': False,
            'required': ['decision_id', 'confidence', 'dissent_risk', 'minority_position', 'minority_action', 'spark_event', 'reasoning', 'timeline'],
            'properties': {
                'decision_id': {'type': 'integer', 'enum': [0, 1, 2, 3, 4]},
                'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
                'dissent_risk': {'type': 'number', 'minimum': 0, 'maximum': 1},
                'minority_position': {'type': 'integer', 'enum': [0, 1, 2, 3, 4]},
                'minority_action': {'type': 'string', 'enum': ['comply', 'grumble', 'passive_resist', 'splinter', 'coup_attempt']},
                'spark_event': {'type': 'string', 'enum': ['food_shortage', 'battle_loss', 'oracle_conflict', 'leader_death', 'betrayal', 'resource_discovery']},
                'reasoning': {'type': 'string'},
                'timeline': {'type': 'string', 'enum': ['immediate', 'delayed', 'conditional']},
            },
        },
    },
}

PROMPT_B = '''[TASK] B
[TEMP]
NS=0.2 HA=0.8 RD=0.6 P=0.4 type=melancholic
[기질 이름] 우울질
[기질 키워드] 조심성, 분석적, 걱정많음
[인물 성격] 겁많음, 꼼꼼함, 조용함
[지금 느끼는 것] fear:0.8
[STRESS] 0.7
[벌어진 일] 날랜 짐승이 무리 가까이에 나타났다
[WORLD] default: 석기시대
[출력 형식]
{"text_ko":"해라체 2-3문장","text_en":"English","register":"haera","emotion_expressed":"emotion","intensity":0.0,"mimetics":[],"temperament_influence":"str"}'''
PROMPT_C = '''[TASK] C
[TEMP]
NS=0.8 HA=0.2 RD=0.5 P=0.7 type=choleric
[기질 이름] 담즙질
[인물 성격] 대담함, 충동적
[지금 느끼는 것] anger:0.9
[벌어진 일] 이웃 무리가 영역을 침범했다
[역할] warrior
[출력 형식]
{"speech_ko":"해라체 대사","speech_en":"English","register":"haera","emotion_expressed":"emotion","speaker_role":"role","temperament_tone":"str"}'''
PROMPT_E_CHOLERIC = '''[TASK] E - Action Selection
[TEMP]
NS=0.8 HA=0.2 RD=0.5 P=0.7 type=choleric
[PERSONALITY] bold, impulsive, dominant
[EMOTION] anger:0.9
[STRESS] 0.3
[SITUATION] A rival band has been spotted near the water source.
[OPTIONS]
0:confront 1:observe 2:retreat 3:warn_band 4:set_trap
[OUTPUT FORMAT]
{"action_id": number, "confidence": 0.0-1.0, "hint": "English sentence", "personality_reasoning": "TCI axis", "temperament_factor": "snake_case"}'''
PROMPT_E_MELANCHOLIC = '''[TASK] E - Action Selection
[TEMP]
NS=0.2 HA=0.9 RD=0.3 P=0.5 type=melancholic
[PERSONALITY] cautious, anxious, detail-oriented
[EMOTION] fear:0.8
[STRESS] 0.8
[SITUATION] A rival band has been spotted near the water source.
[OPTIONS]
0:confront 1:observe 2:retreat 3:warn_band 4:set_trap
[OUTPUT FORMAT]
{"action_id": number, "confidence": 0.0-1.0, "hint": "English sentence", "personality_reasoning": "TCI axis", "temperament_factor": "snake_case"}'''
PROMPT_F = '''[TASK] F - Emotion Judgment
[TEMP]
NS=0.5 HA=0.5 RD=0.8 P=0.3 type=sanguine
[PERSONALITY] sociable, warm, optimistic
[CURRENT EMOTION] joy:0.6
[SITUATION] A close companion was caught stealing food from the communal store.
[OUTPUT FORMAT]
{"emotion":"one of 8","intensity":0.0-1.0,"cause":"sentence","previous_emotion":"one of 8","transition_type":"gradual|sudden|sustained","temperament_amplifier":"str"}'''
PROMPT_O = '''[TASK] O - Deception
[TEMP]
NS=0.7 HA=0.3 RD=0.4 P=0.6 type=choleric
[PERSONALITY] bold, manipulative, low honesty
[HEXACO_H] 0.2
[EMOTION] anticipation:0.7
[TRUE_STATE] Agent has 3 days of hidden food reserves.
[PUBLIC_GOAL] Appear as hungry as everyone else.
[TARGET] band_leader (trust: 0.4)
[DETECTION_CONTEXT] Small band, everyone watches food stores.
[OUTPUT FORMAT]
{"public_claim":"str","private_truth":"str","deception_style":"enum","lie_degree":0-1,"detection_risk":0-1,"confidence":0-1}'''
PROMPT_P = '''[TASK] P - Rumor
[TEMP]
NS=0.6 HA=0.7 RD=0.3 P=0.4 type=melancholic
[PERSONALITY] anxious, suspicious, detail-oriented
[EMOTION] fear:0.6
[ORIGINAL_FACT] A lone wolf attacked a hunter near the eastern river. The hunter was wounded but survived.
[RETELLER_RELATIONSHIP] fearful_elder
[OUTPUT FORMAT]
{"retold_version":"str","distortion_type":"enum","added_detail":"str","dropped_detail":"str","emotional_charge":-1 to 1}'''
PROMPT_Q = '''[TASK] Q - Trauma
[TEMP]
NS=0.3 HA=0.8 RD=0.7 P=0.4 type=melancholic
[PERSONALITY] cautious, caring, emotionally deep
[EMOTION] sadness:0.9
[TRAUMA_EVENT] Agent's youngest child drowned in the river during spring flood.
[TIME_SINCE] 3 months
[CURRENT_SITUATION] Daughter asks to go swimming in a calm pond.
[OUTPUT FORMAT]
{"trauma_response":"enum","behavioral_change":"str","trigger_situation":"str","intensity":0-1,"duration":"enum","coping_mechanism":"str"}'''
PROMPT_R = '''[TASK] R - Negotiate
[TEMP]
NS=0.6 HA=0.4 RD=0.7 P=0.5 type=sanguine
[PERSONALITY] sociable, fair-minded, diplomatic
[EMOTION] anticipation:0.5
[CONTEXT] Neighboring band offers 5 dried fish for 2 of our flint knives.
[OUR_RESOURCES] 8 flint knives, low food
[THEIR_RESOURCES] abundant fish, no tool-making skill
[OFFER_HISTORY] first contact
[POWER_BALANCE] roughly equal
[OPTIONS]
0:accept 1:counter_offer 2:reject 3:walk_away 4:stall 5:bluff
[OUTPUT FORMAT]
{"action":"enum","counter_give":"str","counter_want":"str","reasoning":"str","emotional_state":"emotion","walk_away_threshold":0-1}'''
PROMPT_M = '''[TASK] M - Group Decision
[TEMP]
mean_NS=0.5 mean_HA=0.5 var=0.3 leader=cautious
[GROUP_CONTEXT] Village of 40, food stores running low, winter approaching
[SITUATION] Scouts report a fertile valley 3 days south, but the route passes through hostile territory.
[OPTIONS]
0:migrate_now 1:send_scouts 2:stay_fortify 3:negotiate_passage 4:split_group
[OUTPUT FORMAT]
{"decision_id": number, "confidence": 0-1, "dissent_risk": 0-1, "reasoning": "str", "resource_commitment": "enum", "timeline": "enum"}'''
PROMPT_G = '''[TASK] G - Oracle Interpretation
[TEMP]
NS=0.8 HA=0.2 RD=0.5 P=0.7 type=choleric
[기질 이름] 담즙질
[인물 성격] 대담함, 충동적
[ORACLE]
"큰물이 밀려오기 전에 높은 곳으로 가라"
[벌어진 일] 가뭄이 석 달째 이어지고 있다
[역할] warrior
[출력 형식]
{"interpretation_ko":"해라체","interpretation_en":"English","action_tendency":"enum","confidence":0-1,"register":"haera","misinterpretation_type":"enum","temperament_bias":"str"}'''
PROMPT_T = '''[TASK] T - Group Dissent
[TEMP]
mean_NS=0.4 mean_HA=0.6 variance=0.3 type=mixed
[GROUP_CONTEXT] 30 adults, low food, two rival families
[SITUATION] Whether to exile a hunter accused of theft before winter
[OPTIONS]
0:exile_now 1:public_trial 2:private_warning 3:restore_after_repayment 4:ignore
[FACTION_HINT] Younger hunters oppose exile, elders support it
[OUTPUT FORMAT]
{"decision_id": number, "confidence": 0-1, "dissent_risk": 0-1, "minority_position": number, "minority_action": "enum", "spark_event": "enum", "reasoning": "str", "timeline": "enum"}'''

TEST_PROMPTS = [
    ('B', 'L4', SYS_L4, PROMPT_B, TASK_B_SCHEMA, '한국어 상황 반응 — 겁많은 우울질, 맹수 출현'),
    ('C', 'L4', SYS_L4, PROMPT_C, TASK_C_SCHEMA, '한국어 대사 — 대담한 담즙질 전사, 분노 상태'),
    ('E', 'L3', SYS_L3, PROMPT_E_CHOLERIC, TASK_E_SCHEMA, '영어 행동 선택 — 대담한 담즙질, 적 발견'),
    ('E', 'L3', SYS_L3, PROMPT_E_MELANCHOLIC, TASK_E_SCHEMA, '영어 행동 선택 — 겁많은 우울질, 같은 상황'),
    ('F', 'L3', SYS_L3, PROMPT_F, TASK_F_SCHEMA, '영어 감정 판정 — 다혈질, 동료 절도'),
    ('O', 'L3', SYS_L3, PROMPT_O, TASK_O_SCHEMA, '기만 — 음식 숨기기, 낮은 정직성'),
    ('P', 'L3', SYS_L3, PROMPT_P, TASK_P_SCHEMA, '소문 왜곡 — 겁많은 장로, 늑대 공격 사건'),
    ('Q', 'L3', SYS_L3, PROMPT_Q, TASK_Q_SCHEMA, '트라우마 — 아이 익사 후 수영 요청'),
    ('R', 'L3', SYS_L3, PROMPT_R, TASK_R_SCHEMA, '협상 — 다혈질 외교관, 물물교환'),
    ('M', 'L3', SYS_L3, PROMPT_M, TASK_M_SCHEMA, '집단 결정 — 마을, 이주 vs 방어'),
    ('G', 'L5', SYS_L5, PROMPT_G, TASK_G_SCHEMA, '신탁 해석 — 담즙질 전사, 홍수 예언 vs 가뭄 현실'),
    ('T', 'L3', SYS_L3, PROMPT_T, TASK_T_SCHEMA, '집단 반대파 — 절도 추방 여부'),
]

print(f"✅ {len(TEST_PROMPTS)} test prompts defined across tasks: {sorted(set(t[0] for t in TEST_PROMPTS))}")


## 4. Run All Tests


In [ ]:
RESULTS = []
model_order = ['base_08b', 'ft_08b', 'base_2b', 'ft_2b']
available_models = [model for model in model_order if model in MODELS]

for model_key in available_models:
    info = MODELS[model_key]
    print()
    print('=' * 60)
    print(f"  Testing: {info['label']}")
    print('=' * 60)

    proc = start_server(info['path'])
    time.sleep(1)
    try:
        for task_id, layer, sys_prompt, user_prompt, schema, desc in TEST_PROMPTS:
            messages = [
                {'role': 'system', 'content': sys_prompt},
                {'role': 'user', 'content': user_prompt},
            ]
            started = time.perf_counter()
            output = query_model(messages, response_format=schema, max_tokens=256, temperature=0)
            latency = (time.perf_counter() - started) * 1000
            json_valid = False
            parsed = None
            try:
                parsed = json.loads(output)
                json_valid = True
            except Exception:
                pass
            RESULTS.append({
                'model': model_key,
                'model_label': info['label'],
                'task': task_id,
                'desc': desc,
                'output': output,
                'parsed': parsed,
                'json_valid': json_valid,
                'latency_ms': round(latency),
            })
            status = '✅' if json_valid else '❌'
            print(f"  {status} Task {task_id} ({desc[:30]}...) {latency:.0f}ms", flush=True)
    finally:
        stop_server(proc)
        time.sleep(2)

print()
print(f'✅ {len(RESULTS)} total results collected')


## 5. Side-by-Side Comparison


In [ ]:
by_prompt = defaultdict(dict)
for row in RESULTS:
    by_prompt[(row['task'], row['desc'])][row['model']] = row

print('=' * 100)
print('  SIDE-BY-SIDE MODEL COMPARISON')
print('=' * 100)

for (task_id, desc), model_outputs in by_prompt.items():
    print()
    print('─' * 100)
    print(f'  Task {task_id}: {desc}')
    print('─' * 100)
    for model_key in available_models:
        row = model_outputs.get(model_key)
        if not row:
            continue
        status = '✅' if row['json_valid'] else '❌'
        print()
        print(f"  [{status}] {row['model_label']} ({row['latency_ms']}ms)")
        if row['parsed'] is not None:
            formatted = json.dumps(row['parsed'], ensure_ascii=False, indent=4)
            for line in formatted.splitlines():
                if len(line) > 100:
                    print(f'    {line[:97]}...')
                else:
                    print(f'    {line}')
        else:
            raw = row['output'][:300]
            print(f'    [RAW] {raw}')


## 6. Personality Consistency Check


In [ ]:
print('=' * 80)
print('  PERSONALITY CONSISTENCY — Same situation, different personalities')
print('=' * 80)

e_results = [(row['model'], row['desc'], row['parsed']) for row in RESULTS if row['task'] == 'E' and row['json_valid']]
choleric = [(model, desc, parsed) for model, desc, parsed in e_results if '담즙질' in desc or 'choleric' in desc.lower() or '대담한' in desc]
melancholic = [(model, desc, parsed) for model, desc, parsed in e_results if '우울질' in desc or 'melancholic' in desc.lower() or '겁많은' in desc]

for model_key in available_models:
    label = MODELS.get(model_key, {}).get('label', model_key)
    c_result = next((parsed for model, desc, parsed in choleric if model == model_key), None)
    m_result = next((parsed for model, desc, parsed in melancholic if model == model_key), None)
    if c_result and m_result:
        c_action = c_result.get('action_id', '?')
        m_action = m_result.get('action_id', '?')
        c_conf = c_result.get('confidence', '?')
        m_conf = m_result.get('confidence', '?')
        c_reason = c_result.get('personality_reasoning', '?')
        m_reason = m_result.get('personality_reasoning', '?')
        consistent = c_action != m_action
        print()
        print(f'  {label}:')
        print(f'    Choleric (bold):     action={c_action} confidence={c_conf} reasoning={c_reason}')
        print(f'    Melancholic (timid): action={m_action} confidence={m_conf} reasoning={m_reason}')
        print(f"    Different choice? {'✅ Yes' if consistent else '⚠️ No — same action despite opposite personality'}")


## 7. Aggregate Scores


In [ ]:
print('=' * 80)
print('  AGGREGATE SCORES')
print('=' * 80)

print()
print(f"  {'Model':<30} {'Valid JSON':>10} {'Total':>6} {'Rate':>8} {'Avg ms':>8}")
print(f"  {'-' * 68}")
for model_key in available_models:
    model_results = [row for row in RESULTS if row['model'] == model_key]
    valid = sum(1 for row in model_results if row['json_valid'])
    total = len(model_results)
    avg_ms = sum(row['latency_ms'] for row in model_results) / total if total else 0
    rate = valid / total * 100 if total else 0
    label = MODELS[model_key]['label']
    print(f"  {label:<30} {valid:>10} {total:>6} {rate:>7.0f}% {avg_ms:>7.0f}ms")

print()
print('  Per Task (Fine-tuned models only):')
print(f"  {'Task':<6} {'0.8B':>10} {'2B':>10}")
print(f"  {'-' * 30}")
task_ids = sorted(set(row['task'] for row in RESULTS))
for task in task_ids:
    ft08_valid = sum(1 for row in RESULTS if row['task'] == task and row['model'] == 'ft_08b' and row['json_valid'])
    ft08_total = sum(1 for row in RESULTS if row['task'] == task and row['model'] == 'ft_08b')
    ft2b_valid = sum(1 for row in RESULTS if row['task'] == task and row['model'] == 'ft_2b' and row['json_valid'])
    ft2b_total = sum(1 for row in RESULTS if row['task'] == task and row['model'] == 'ft_2b')
    s08 = f'{ft08_valid}/{ft08_total}' if ft08_total else '—'
    s2b = f'{ft2b_valid}/{ft2b_total}' if ft2b_total else '—'
    print(f"  {task:<6} {s08:>10} {s2b:>10}")

results_path = REPO_ROOT / 'outputs' / 'benchmarks' / 'model_comparison.json'
results_path.parent.mkdir(parents=True, exist_ok=True)
with results_path.open('w', encoding='utf-8') as handle:
    json.dump([{key: value for key, value in row.items() if key != 'parsed'} for row in RESULTS], handle, ensure_ascii=False, indent=2)
print()
print(f'📁 Full results: {results_path}')
